[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-08-secrets-variables.ipynb#scrollTo=10a2b3c4)

---
# Day 8 · Secrets, Variables, and Environment Configuration
**certified-journeys / kestra-certified** · Week 2 · Secure Configuration

> **Goal for today:** Store secrets safely using `{{ secret('NAME') }}`, define namespace-level variables, use Pebble filters like `| upper`, and parameterize flows for dev/staging/prod without duplicating flow YAML.


In [ ]:
%pip install -q pyyaml


## Step 1 · Kestra Secrets — Never Hardcode Credentials

Kestra stores secrets **server-side, encrypted at rest**. They are:
- Created via the Kestra UI (`Settings → Secrets`) or via the API
- Referenced in flow YAML with `{{ secret('SECRET_NAME') }}`
- **Masked in all execution logs** — Kestra redacts the value automatically
- Scoped to a namespace (a secret in `data.ingestion` is not visible in `data.reporting` unless explicitly shared)

**Anti-patterns to avoid:**

| ❌ Wrong | ✅ Right |
|---|---|
| `password: mypassword123` | `password: "{{ secret('DB_PASSWORD') }}"` |
| `apiKey: sk-abc123` | `apiKey: "{{ secret('OPENAI_API_KEY') }}"` |
| Store in flow inputs with `type: STRING` | Use `type: SECRET` input type |

> The `type: SECRET` input type also masks the value in UI forms — use it whenever an input should not be logged.


In [ ]:
import yaml

# ── Flow using secrets for an API call ───────────────────────────────────────
flow_with_secrets = {
    "id": "fetch-api-data",
    "namespace": "data.ingestion",
    "tasks": [
        {
            "id": "call-api",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://api.example.com/data",
            "method": "GET",
            "headers": {
                # secret() is resolved server-side; the value never appears in YAML
                "Authorization": "Bearer {{ secret('MY_API_KEY') }}",
                "X-Tenant-ID":   "{{ secret('TENANT_ID') }}",
            },
        },
        {
            "id": "log-status",
            "type": "io.kestra.plugin.core.log.Log",
            # Safe to log status code — not the secret value
            "message": "API responded with status: {{ outputs['call-api'].code }}",
        },
    ],
}

print("=== Flow with Secrets (safe to commit to git) ===")
print(yaml.dump(flow_with_secrets, default_flow_style=False, sort_keys=False))

# ── Simulate what Kestra does at runtime ─────────────────────────────────────
# (In a real Kestra instance, secret() is a server-side function, not Python)
def secret(name):
    """Simulates the Kestra secret() function. Returns masked output for demo."""
    fake_store = {
        "MY_API_KEY": "YOUR_API_KEY_HERE",   # placeholder — never real values
        "TENANT_ID":  "tenant-42",
        "DB_PASSWORD": "YOUR_DB_PASSWORD_HERE",
    }
    value = fake_store.get(name, f"<SECRET:{name}_NOT_FOUND>")
    return "*" * len(value)  # Kestra masks this in logs

print("Simulated log output (secrets masked):")
print(f"  Authorization: Bearer {secret('MY_API_KEY')}")
print(f"  X-Tenant-ID:   {secret('TENANT_ID')}")


**What just happened?**
- The flow YAML is **safe to commit to git** — it only contains `{{ secret('MY_API_KEY') }}`, not the actual key.
- **Kestra masks** the resolved secret value in execution logs — shown above as `****`.
- Our Python `secret()` simulation mimics the masking behavior for demonstration.
- The `SECRET` type input (not shown here but covered next) also prevents the value from appearing in Kestra's UI forms.


## Step 2 · Kestra Variables — Namespace-Level Configuration

**Variables** are key-value pairs stored at the namespace level. Unlike secrets, they are **not encrypted** — use them for non-sensitive config like:
- Base URLs (`base_url: https://api.example.com`)
- Dataset paths (`raw_bucket: gs://my-bucket/raw`)
- Thresholds (`max_null_pct: 0.05`)

They are defined in the Kestra UI under `Namespaces → <namespace> → Variables`, and referenced with:
```
{{ vars.variable_name }}
```

**Inheritance:** A variable defined at `data` is inherited by `data.ingestion`, `data.transformation`, etc. — unless overridden at a more specific level.

| Scope | Example use |
|---|---|
| Root `data` namespace | `env: production` shared by all sub-namespaces |
| `data.ingestion` | `source_bucket: gs://raw-data` specific to ingestion |
| `data.reporting` | `report_recipients: ops@example.com` |


In [ ]:
# ── Flow referencing namespace-level variables ─────────────────────────────
flow_with_vars = {
    "id": "ingest-from-bucket",
    "namespace": "data.ingestion",
    "tasks": [
        {
            "id": "log-config",
            "type": "io.kestra.plugin.core.log.Log",
            # vars.* are namespace variables; no secret masking applied
            "message": (
                "Running in env={{ vars.env }} | "
                "bucket={{ vars.source_bucket }} | "
                "max_null_pct={{ vars.max_null_pct }}"
            ),
        },
        {
            "id": "download",
            "type": "io.kestra.plugin.core.http.Download",
            # Pebble: vars.base_url is concatenated with a static path
            "uri": "{{ vars.base_url }}/daily/addresses.csv",
        },
    ],
}

print("=== Flow referencing namespace variables ===")
print(yaml.dump(flow_with_vars, default_flow_style=False, sort_keys=False))

# ── Simulate variable resolution for two environments ────────────────────────
namespace_vars = {
    "dev": {
        "env": "dev",
        "base_url": "https://dev-api.example.com",
        "source_bucket": "gs://dev-raw-data",
        "max_null_pct": 0.10,
    },
    "prod": {
        "env": "production",
        "base_url": "https://api.example.com",
        "source_bucket": "gs://prod-raw-data",
        "max_null_pct": 0.02,
    },
}

template = (
    "Running in env={{ vars.env }} | "
    "bucket={{ vars.source_bucket }} | "
    "max_null_pct={{ vars.max_null_pct }}"
)

print("\nSimulated resolution per environment:")
for env_name, vars_ in namespace_vars.items():
    rendered = template
    for k, v in vars_.items():
        rendered = rendered.replace("{{ vars." + k + " }}", str(v))
    print(f"  [{env_name}] {rendered}")


**What just happened?**
- **`vars.env`**, **`vars.source_bucket`** — namespace variables resolve to different values in dev vs prod without changing the flow YAML.
- We simulated the Pebble resolver in Python: for each environment, substitute `{{ vars.key }}` → actual value.
- **One flow YAML serves all environments** — the only thing that changes is the namespace variable values, set in the Kestra UI per namespace.


## Step 3 · Pebble Template Expressions and Filters

Kestra uses the **Pebble template engine** for all `{{ }}` expressions. Pebble provides:
- **Variable access:** `{{ vars.env }}`, `{{ inputs.source_url }}`, `{{ secret('KEY') }}`
- **Filters:** transform values — `{{ vars.env | upper }}` → `PRODUCTION`
- **Functions:** `now()`, `dateAdd()`, `render()`
- **Control flow:** `{% if %}`, `{% for %}` (use sparingly — Kestra handles branching with dedicated tasks)

**Commonly used filters:**

| Filter | Example | Result |
|---|---|---|
| `upper` | `{{ vars.env \| upper }}` | `PRODUCTION` |
| `lower` | `{{ inputs.name \| lower }}` | `addresses` |
| `split` | `{{ url \| split('/') \| last }}` | `file.csv` |
| `replace` | `{{ vars.env \| replace({dev: staging}) }}` | `staging` |
| `default` | `{{ inputs.format \| default('csv') }}` | `csv` (if input absent) |
| `date` | `{{ now() \| date('yyyy-MM-dd') }}` | `2026-06-07` |


In [ ]:
# ── Demonstrate Pebble filter patterns in Python ─────────────────────────────
from datetime import datetime

# Simulate Pebble filter: | upper
def pebble_upper(val):
    return val.upper()

# Simulate Pebble filter: | split('/') | last
def pebble_split_last(val, sep="/"):
    return val.split(sep)[-1]

# Simulate Pebble filter: | default('csv')
def pebble_default(val, fallback):
    return val if val is not None and val != "" else fallback

# Simulate Pebble: now() | date('yyyy-MM-dd')
def pebble_date_now(fmt="%Y-%m-%d"):
    return datetime.now().strftime(fmt)

examples = {
    "{{ vars.env | upper }}": pebble_upper("production"),
    "{{ url | split('/') | last }}": pebble_split_last(
        "https://example.com/data/addresses.csv"
    ),
    "{{ inputs.format | default('csv') }}": pebble_default(None, "csv"),
    "{{ now() | date('yyyy-MM-dd') }}": pebble_date_now(),
}

print("Pebble filter examples:")
for expr, result in examples.items():
    print(f"  {expr:45s} → {result}")

# ── Flow using filters for dynamic file naming ────────────────────────────────
flow_pebble_filters = {
    "id": "export-with-date",
    "namespace": "data.reporting",
    "tasks": [
        {
            "id": "log-export",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                # Dynamic filename: report_PRODUCTION_2026-06-07.csv
                "Exporting: report_{{ vars.env | upper }}_"
                "{{ now() | date('yyyy-MM-dd') }}.csv"
            ),
        }
    ],
}
print("\nFlow with Pebble filters:")
print(yaml.dump(flow_pebble_filters, default_flow_style=False, sort_keys=False))


**What just happened?**
- We implemented Python analogues for the four most common Pebble filters to show the transformation logic.
- **`| upper`** is commonly used for environment names in file paths and log messages.
- **`| default('csv')`** prevents null pointer errors when an optional input is not provided.
- **`now() | date('yyyy-MM-dd')`** generates date-stamped output file names — critical for idempotent pipelines.


## Step 4 · Environment Variables in Script Tasks

When running script tasks (`io.kestra.plugin.scripts.python.Script`), you can inject values — including secrets and variables — as **environment variables** inside the process.

```yaml
type: io.kestra.plugin.scripts.python.Script
env:
  API_KEY: "{{ secret('MY_API_KEY') }}"
  ENV_NAME: "{{ vars.env }}"
script: |
  import os
  api_key = os.environ['API_KEY']   # masked in logs
  env     = os.environ['ENV_NAME']  # visible in logs
```

This pattern is preferred over embedding `{{ secret() }}` directly in the script body — it keeps scripts portable and testable outside Kestra.


In [ ]:
import os

# ── Generate the flow YAML ────────────────────────────────────────────────────
flow_env_vars = {
    "id": "script-with-env-vars",
    "namespace": "data.ingestion",
    "tasks": [
        {
            "id": "fetch-data",
            "type": "io.kestra.plugin.scripts.python.Script",
            "env": {
                # Secrets injected as env vars — masked in execution logs
                "API_KEY":      "{{ secret('MY_API_KEY') }}",
                "DB_PASSWORD":  "{{ secret('DB_PASSWORD') }}",
                # Non-sensitive config from namespace variables
                "ENV_NAME":     "{{ vars.env }}",
                "BASE_URL":     "{{ vars.base_url }}",
            },
            "script": (
                "import os\n"
                "import json\n\n"
                "api_key  = os.environ['API_KEY']\n"
                "env_name = os.environ['ENV_NAME']\n"
                "base_url = os.environ['BASE_URL']\n\n"
                "# api_key is masked by Kestra in logs — safe to use\n"
                "print(f'Connecting to {base_url} in {env_name}')\n"
                "print(f'API key length: {len(api_key)} chars')  # log length, not value\n"
            ),
        }
    ],
}

print("=== Script task with env vars ===")
print(yaml.dump(flow_env_vars, default_flow_style=False, sort_keys=False))

# ── Simulate the script running locally with placeholder values ───────────────
print("=== Simulated local script execution ===")
# Set placeholder env vars (as Kestra would inject them)
os.environ["API_KEY"]  = "YOUR_API_KEY_HERE"   # placeholder
os.environ["ENV_NAME"] = "dev"
os.environ["BASE_URL"] = "https://dev-api.example.com"

api_key  = os.environ["API_KEY"]
env_name = os.environ["ENV_NAME"]
base_url = os.environ["BASE_URL"]

print(f"Connecting to {base_url} in {env_name}")
print(f"API key length: {len(api_key)} chars")


**What just happened?**
- The `env:` block injects both **secrets** (masked) and **variables** (visible) as environment variables into the Python process.
- Inside the script, `os.environ['API_KEY']` reads the injected value — the script itself doesn't need to know anything about Kestra's secret store.
- **Log the length, not the value** — a safe logging pattern when working with API keys.


## Step 5 · Parameterizing a Flow for dev / staging / prod

The pattern: one flow YAML, one `ENV` input, namespace variables set per environment.

```
Trigger flow with inputs.ENV = "dev"
    ↓
Pebble: {{ vars.base_url }}  ← resolves from the *current namespace's* variables
    ↓
Different behavior per env without changing flow YAML
```

Alternatively, use separate namespaces per environment:
- `data.ingestion.dev`
- `data.ingestion.staging`
- `data.ingestion.prod`

Each namespace has its own variable set — deploy the same flow YAML to all three.


In [ ]:
# ── Multi-env parameterized flow ──────────────────────────────────────────────
multi_env_flow = {
    "id": "parameterized-pipeline",
    "namespace": "data.ingestion",
    "inputs": [
        {
            "id": "ENV",
            "type": "SELECT",
            "values": ["dev", "staging", "prod"],
            "defaults": "dev",
            "description": "Target environment for this run.",
        }
    ],
    "tasks": [
        {
            "id": "log-environment",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Starting pipeline | env={{ inputs.ENV | upper }} | "
                "base_url={{ vars.base_url }}"
            ),
        },
        {
            "id": "download",
            "type": "io.kestra.plugin.core.http.Download",
            "uri": "{{ vars.base_url }}/data/daily.csv",
        },
        {
            "id": "process",
            "type": "io.kestra.plugin.scripts.python.Script",
            "env": {
                "DB_PASSWORD": "{{ secret('DB_PASSWORD') }}",
                "ENV_NAME":    "{{ inputs.ENV }}",
            },
            "script": (
                "import os\n"
                "env = os.environ['ENV_NAME']\n"
                "print(f'Processing for environment: {env}')\n"
            ),
        },
    ],
}

print("=== Multi-environment parameterized flow ===")
print(yaml.dump(multi_env_flow, default_flow_style=False, sort_keys=False))

# Show how the same flow resolves differently per environment
print("=== Resolution per environment ===")
env_configs = [
    {"ENV": "dev",     "vars": {"base_url": "https://dev-api.example.com"}},
    {"ENV": "staging", "vars": {"base_url": "https://staging-api.example.com"}},
    {"ENV": "prod",    "vars": {"base_url": "https://api.example.com"}},
]

for cfg in env_configs:
    env_label = cfg["ENV"].upper()
    base_url  = cfg["vars"]["base_url"]
    uri       = f"{base_url}/data/daily.csv"
    print(f"  ENV={env_label:8s} → download URI: {uri}")


**What just happened?**
- A single `SELECT` input `ENV` lets the user pick the environment in the Kestra UI before triggering.
- **`{{ vars.base_url }}`** resolves from the namespace's variable store — the same flow YAML deployed to `data.ingestion.dev` vs `data.ingestion.prod` resolves to different URLs.
- The table shows that `dev`, `staging`, and `prod` produce three different download URIs — zero code duplication.


In [ ]:
# Challenge: Complete the secure flow below.
#
# Add a Python script task that:
#   1. Reads DB_HOST from a namespace variable {{ vars.db_host }}
#   2. Reads DB_PASSWORD from a secret {{ secret('DB_PASSWORD') }}
#      (inject both as env vars in the 'env:' block)
#   3. Inside the script: prints the host, and prints the password LENGTH (not value)
#
# Bonus: add a Pebble filter so the env name is always uppercased in log output.
#
# Your solution here:
challenge_flow = {
    "id": "secure-db-connect",
    "namespace": "data.ingestion",
    "tasks": [
        {
            "id": "connect",
            "type": "io.kestra.plugin.scripts.python.Script",
            # TODO: add env block with DB_HOST and DB_PASSWORD
            "env": {},
            # TODO: write the script body
            "script": "print('TODO: implement me')",
        }
    ],
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `{{ secret('NAME') }}` | Server-side secret lookup — value is **masked in all logs** |
| `type: SECRET` input | Masks value in the Kestra UI trigger form |
| `{{ vars.name }}` | Namespace-level variable — visible in logs, not for credentials |
| Variable inheritance | Parent namespace vars are available to child namespaces |
| Pebble `| upper` | Transforms value to uppercase — use for env labels in file names |
| Pebble `| default(x)` | Fallback when an optional input is absent |
| `env:` block in scripts | Injects secrets/vars as OS environment variables — portable |
| Multi-env strategy | One YAML + namespace variables per env = zero duplication |

> **Tip:** Never hardcode credentials in flow YAML. Use `{{ secret('NAME') }}` for sensitive values — both are masked in execution logs.

---
## What's next
**Day 9** → Python Scripts — Tasks, Docker, and Working with Data — run pandas inside Kestra, pass outputs between tasks, and choose between PROCESS and DOCKER runners.

Mark Day 8 complete in your [tracker](../index.html).
